# Lab 09-02 · Carriers, Doping and the Fermi Level

Companion to chapter [09-02](../../chapters/09-02-carriers-doping-fermi-level.md). Handout: [README.md](README.md).

The notebook computes carrier statistics without the Boltzmann shortcut: Fermi–Dirac integrals by quadrature, and a general charge-neutrality solver that finds the Fermi level by bisection with donor and acceptor ionization. Each part asserts agreement with the chapter.

In [1]:
import math
import numpy as np
from scipy.integrate import quad
from scipy.optimize import brentq
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

K = 8.617333262e-5                     # eV/K
NC300, NV300, NI300 = 3.2e19, 1.8e19, 1.0e10
def F12(eta):
    return 2 / math.sqrt(math.pi) * quad(lambda x: math.sqrt(x) / (1 + math.exp(min(x - eta, 700))),
                                         0, max(eta, 0) + 60, epsabs=0, epsrel=1e-11)[0]
print("F12(0) =", round(F12(0.0), 5))

F12(0) = 0.76515


## Part 1 — When does Boltzmann fail?

Compare $n = N_C\mathcal F_{1/2}(\eta)$ with the Boltzmann value $N_Ce^\eta$ as the Fermi level approaches and enters the band.

**🔮 Predict:** at what $\eta = (E_F - E_C)/kT$ does the Boltzmann value overstate $n$ by 5%? (The chapter's rule of thumb is $3kT$, i.e. $\eta = -3$.)

In [2]:
etas = np.linspace(-6, 4, 201)
ratio = np.array([F12(e) / math.exp(e) for e in etas])
eta5 = brentq(lambda e: F12(e) / math.exp(e) - 0.95, -10, 5)
print(f"Boltzmann overstates n by 5% at eta = {eta5:.3f}, i.e. N_D = {NC300 * F12(eta5):.3e} cm^-3")
fig, ax = plt.subplots(figsize=(5.4, 3.4))
ax.plot(etas, ratio); ax.axhline(0.95, ls=":", c="0.5"); ax.axvline(-3, ls="--", c="0.5")
ax.set_xlabel("η = (E_F − E_C)/kT"); ax.set_ylabel("F½(η) / e^η")
fig.tight_layout(); plt.close(fig)
assert abs(eta5 - (-1.877)) < 5e-3

Boltzmann overstates n by 5% at eta = -1.877, i.e. N_D = 4.652e+18 cm^-3


## Part 2 — A general charge-neutrality solver

Find $E_F$ (measured from $E_V$) such that $p + N_D^+ - n - N_A^- = 0$, with Fermi–Dirac carriers and ionization factors $g_D = 2$, $g_A = 4$.

**🔮 Predict (the chapter's guided exercise):** is 10¹⁷ cm⁻³ phosphorus fully ionized at 300 K? Estimate the ionized fraction, and the temperature at which half the donors freeze out.

In [3]:
def solve(ND=0.0, NA=0.0, T=300.0, dED=0.045, dEA=0.045, Eg=1.12):
    kT = K * T
    NC = NC300 * (T / 300) ** 1.5
    NV = NV300 * (T / 300) ** 1.5
    def parts(EF):
        n = NC * F12((EF - Eg) / kT)
        p = NV * F12((0.0 - EF) / kT)
        ndp = ND / (1 + 2 * math.exp(min((EF - (Eg - dED)) / kT, 700))) if ND else 0.0
        nam = NA / (1 + 4 * math.exp(min((dEA - EF) / kT, 700))) if NA else 0.0
        return n, p, ndp, nam
    def charge(EF):
        n, p, ndp, nam = parts(EF)
        return p + ndp - n - nam
    EF = brentq(charge, -0.5, Eg + 0.5, xtol=1e-12)
    return (EF,) + parts(EF)

EF, n, p, ndp, _ = solve(ND=1e16, T=300)
print(f"1e16 P, 300 K: n = {n:.4e}, E_C - E_F = {1.12 - EF:.4f} eV, ionized {ndp/1e16:.4f}")
assert abs(n - 9.965e15) < 5e11
EF, n, p, ndp, _ = solve(ND=1e16, T=77)
print(f"1e16 P, 77 K:  n = {n:.4e}, ionized {ndp/1e16:.4f}")
assert abs(ndp / 1e16 - 0.382) < 5e-4
EF, n, p, ndp, _ = solve(ND=1e17, T=300)
frac17 = ndp / 1e17
T50 = brentq(lambda T: solve(ND=1e17, T=T)[3] / 1e17 - 0.5, 40, 300)
print(f"1e17 P, 300 K: ionized {frac17:.4f}; half frozen out at {T50:.1f} K")
assert abs(frac17 - 0.9667) < 5e-4 and abs(T50 - 119.2) < 0.1

1e16 P, 300 K: n = 9.9646e+15, E_C - E_F = 0.2087 eV, ionized 0.9965
1e16 P, 77 K:  n = 3.8186e+15, ionized 0.3819


1e17 P, 300 K: ionized 0.9667; half frozen out at 119.2 K


In [4]:
# Degenerate doping (Example 09-02.7): treat the donors as fully ionized (impurity band)
for ND in (1e19, 1e20):
    EF, n, *_ = solve(ND=ND, T=300, dED=-5.0)       # a donor level far above E_C is always empty
    print(f"N_D = {ND:.0e}: E_F - E_C = {(EF - 1.12)*1e3:+.1f} meV")
EF19 = solve(ND=1e19, T=300, dED=-5.0)[0]
assert abs((1.12 - EF19) * 1e3 - 27.2) < 0.06

N_D = 1e+19: E_F - E_C = -27.2 meV
N_D = 1e+20: E_F - E_C = +56.9 meV


## Part 3 — Fermi level against doping and carriers against temperature

Reproduce chapter Figures 2 and 3. For the temperature sweep, use the course $n_i(T)$ (Varshni gap, $T^{3/2}$ prefactor scaled to $10^{10}$ cm⁻³ at 300 K) for holes and the neutrality equation for electrons.

In [5]:
def Eg_T(T): return 1.17 - 4.73e-4 * T**2 / (T + 636)
def ni_T(T): return NI300 * (T / 300) ** 1.5 * math.exp(Eg_T(300) / (2 * K * 300) - Eg_T(T) / (2 * K * T))
def n_of_T(T, ND=1e15, dE=0.045):
    kT = K * T; NC = NC300 * (T / 300) ** 1.5; ni = ni_T(T)
    def charge(u):
        n = NC * math.exp(u / kT)
        return ni**2 / max(n, 1e-300) + ND / (1 + 2 * math.exp((u + dE) / kT)) - n
    return NC * math.exp(brentq(charge, -1.5, 0.2) / kT)
Ts = np.linspace(25, 700, 300)
ns = np.array([n_of_T(t) for t in Ts])
T_int = brentq(lambda T: ni_T(T) - 1e15, 300, 900)
T_half = brentq(lambda T: n_of_T(T) - 0.5e15, 30, 150)
print(f"n_i = N_D at {T_int:.1f} K; n = N_D/2 at {T_half:.1f} K")
fig, ax = plt.subplots(figsize=(5.6, 3.6))
ax.semilogy(Ts, ns, label="n"); ax.semilogy(Ts, [ni_T(t) for t in Ts], "--", label="n_i")
ax.set_ylim(1e12, 1e17); ax.legend(); ax.set_xlabel("T (K)"); ax.set_ylabel("cm^-3")
fig.tight_layout(); plt.close(fig)
assert abs(T_int - 547.2) < 0.1 and abs(T_half - 64.7) < 0.3

n_i = N_D at 547.2 K; n = N_D/2 at 64.7 K


## Part 4 — Counting dopants

Draw Poisson samples for the dopant count in a channel volume and compare the spread with $\sqrt{NV}$.

In [6]:
rng = np.random.default_rng(1)
for label, V, N in (("planar 30x30x10 nm, 1e18", 30e-7 * 30e-7 * 10e-7, 1e18),
                    ("fin 7x40x20 nm, 1e17", 7e-7 * 40e-7 * 20e-7, 1e17)):
    counts = rng.poisson(N * V, 200_000)
    print(f"{label}: mean {counts.mean():.3f}, sd {counts.std():.3f}, P(0) {np.mean(counts == 0):.3f}")
counts = rng.poisson(0.56, 200_000)
assert abs(np.mean(counts == 0) - math.exp(-0.56)) < 0.005

planar 30x30x10 nm, 1e18: mean 8.995, sd 2.996, P(0) 0.000


fin 7x40x20 nm, 1e17: mean 0.556, sd 0.744, P(0) 0.573


## Wrap-up

- The Boltzmann value of $n$ is within 5% until $E_F$ is about $1.9kT$ below the band edge; the chapter's $3kT$ rule is a conservative bound on the occupation of individual states.
- One neutrality solver covers doped, compensated, partly ionized and degenerate material.
- Freeze-out and intrinsic onset bound a temperature window that depends on doping; lightly doped regions hit the top of the window first.
- Small volumes hold few dopants; the Poisson spread motivates undoped channels.